# CottonLens: gerçek veriyle kısa çalışma pilotu

Bu defter, mevcut `research-free-data-v1` deneyinde **yalnız base feature, XGBoost, T+1, fold 1, aday 1** için üç geçmiş iç doğrulama bloğunu çalıştırır. Yaklaşık altı fit beklenir. **Tümünü çalıştır** geniş `ablate`/`search` turunu başlatmaz. Tamamlanan fitler checksum ile Drive’a kaydedilir; kesilirse aynı defter ve aynı deney kimliğiyle devam edilir.

Bu bir **süre, GPU ve resume pilotudur**; tek aday/tek fold sonucu model üstünlüğü veya seçim kanıtı değildir. Yeni USDA/ECB verileri bu deneye dahil değildir. Writer kilidi varsa eski Colab oturumu durmadan silmeyin.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'research-free-data-v1'  # Retain this identifier for resumes of EXACTLY this data/source.


In [ ]:
import hashlib, importlib, json, sys, zipfile
from pathlib import PurePosixPath
SOURCE_ZIP = DRIVE_ROOT / 'sources/cottonlens-free-research-source-v1.zip'
EXPECTED_SOURCE_ID = 'c669ca9a9d4db2647b80993881efda2ba65b050ea26c6fd16a06a5c1e5dd1b93'
EXPECTED_ZIP_SHA256 = 'ff4eec811ef9549e7311a467475cca2b7826044566aecf5704b792668b06c6b1'
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert expected_zip == EXPECTED_ZIP_SHA256, 'Source ZIP sidecar differs from pinned bundle'
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_ZIP_SHA256, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'] == EXPECTED_SOURCE_ID, 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
PYTHON = setup(research=True)  # Locked managed Python 3.12.11; no global packages.
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
import json
from colab_setup import process_env

assert identity == EXPECTED_SOURCE_ID, 'Wrong source bundle; use the pinned free-research ZIP'
folder = DRIVE_ROOT / 'experiments' / EXPERIMENT
ready_path = folder / 'ready.json'
assert ready_path.is_file(), 'Prepared experiment missing; do not create a new experiment here'
ready = json.loads(ready_path.read_text(encoding='utf-8'))
assert ready['identity']['source_id'] == identity, 'Experiment and source bundle differ'
assert ready['identity']['profile'] == 'free-data-v1', 'Wrong experiment profile'
assert len(ready['identity']['split']['folds']) == 8, 'Expected eight frozen historical folds'
assert not ready['identity']['publication_sources'], 'Unexpected new data package: review before training'
assert not (folder / '.writer-lock').exists(), 'Writer lock present: verify old Colab session stopped before recovery'
BASE = [PYTHON, '-m', 'cottonlens_ml.research.engine', '--repo', REPO,
        '--drive-root', DRIVE_ROOT, '--experiment', EXPERIMENT]
run([*BASE, '--stage', 'status'])  # Read-only inventory; no model fit.
print('Pinned source and prepared cohort match. Next cell runs the eight small diagnostic fits.')


In [ ]:
import json
import colab_progress
from IPython.display import display, Markdown

DIAGNOSIS = json.loads((folder / 'diagnosis.json').read_text(encoding='utf-8'))
assert DIAGNOSIS['status'] == 'passed', 'The GPU diagnostic gate must pass first'
PILOT = folder / 'pilots' / 'base-xgboost-t1-fold1-candidate1.json'
completed_dir = folder / 'ledger' / 'completed'
saved = len(list(completed_dir.glob('*.json')))
assert 8 <= saved <= 14, f'Unexpected completed-fit count: {saved}; inspect the experiment before continuing'
panel = display(Markdown('Checking the pilot; up to six real-data fits.'), display_id=True)
def update(message):
    panel.update(Markdown('```text\n' + message + '\n```'))
pilot_program = "import json\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom cottonlens_ml.code_identity import digest\nfrom cottonlens_ml.research.engine import Experiment\nfrom cottonlens_ml.research.search import recipes\nfrom cottonlens_ml.research.ledger import freeze_record, read_record\nfrom cottonlens_ml.runtime_guard import require_colab_training\nfrom cottonlens_ml.sprint import writer\n\nrequire_colab_training()\nroot, repo = Path(sys.argv[1]), Path(sys.argv[2])\nexperiment = Experiment(root, repo=repo)\nassert experiment.identity['profile'] == 'free-data-v1'\ndiagnosis = read_record(root / 'diagnosis.json')\nassert diagnosis['status'] == 'passed'\ndiagnostic_ids = {control['record'] for control in diagnosis['controls'].values()}\nfold = experiment.identity['split']['folds'][0]\nassert fold['fold'] == 1\nspec = recipes('xgboost', 1, experiment.identity['groups']['base'], 16)[0]\nmarker = root / 'pilots' / 'base-xgboost-t1-fold1-candidate1.json'\nwith writer(root):\n    if marker.exists():\n        result = read_record(marker)\n        assert result['recipe'] == spec and result['fold'] == 1\n        print('PILOT cached completed record; no new fits', flush=True)\n    else:\n        started = time.monotonic()\n        score = experiment.inner(spec, fold)\n        pilot_ids = sorted({p.stem for p in (root / 'ledger' / 'completed').glob('*.json')} - diagnostic_ids)\n        assert len(pilot_ids) == 6, f'Expected six pilot fits, found {len(pilot_ids)}'\n        result = {'purpose': 'runtime_and_resume_pilot_not_model_selection',\n                  'source_id': experiment.identity['source_id'],\n                  'group': 'base', 'family': 'xgboost', 'horizon': 1,\n                  'fold': 1, 'candidate': 1, 'recipe': spec,\n                  'inner_relative_price_mae': score['score'],\n                  'checkpoint_ids': pilot_ids}\n        freeze_record(marker, result)\n        print(f'PILOT saved in {time.monotonic() - started:.1f} seconds', flush=True)\n    assert len(result['checkpoint_ids']) == 6\n    for key in result['checkpoint_ids']:\n        record = read_record(root / 'ledger' / 'completed' / (key + '.json'))\n        for name, checksum in record['files'].items():\n            assert digest(root / 'ledger' / name) == checksum, f'Corrupt pilot checkpoint: {key}'\n    print('PILOT result:', json.dumps({'inner_relative_price_mae': result['inner_relative_price_mae'],\n                                     'fold': result['fold'], 'candidate': result['candidate'],\n                                     'verified_checkpoints': len(result['checkpoint_ids'])}), flush=True)\n"
colab_progress.supervise(
    [PYTHON, '-c', pilot_program, str(folder), str(REPO)],
    cwd=REPO, env=process_env(), drive_logs=DRIVE_ROOT / 'reports',
    decisions=completed_dir, decision_pattern='*.json', total=14, update=update,
)
result = json.loads(PILOT.read_text(encoding='utf-8'))
print('Pilot inner relative price-MAE:', result['inner_relative_price_mae'])
print('Verified pilot checkpoints:', len(result['checkpoint_ids']))
print('A score below 1 means this single candidate beat Naive on these inner blocks only.')
print('Do not use this single-fold score for model selection; share the final panel and result.')
run([*BASE, '--stage', 'status'])
